<a href="https://colab.research.google.com/github/dee0742/ML-FlyRank-Task/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Content Growth vs. Decline (The Anatomy of Growing Content)
- Paper Finding: Content trending upward (growing) has a distinct structural profile compared to content trending downward (declining). Specifically, growing pages are 37.6% longer in word count (3.2K vs. 2.3K words) and 20% younger (184 vs. 230 days) on average.  
- Label Origin: The label for this finding is derived using a heuristic threshold rule. The outcome category (Up vs. Down) is calculated from a 30-day versus previous 30-day impression change window. Content is categorized as Up if it exhibits $>10\%$ growth, and Down if it exhibits $>10\%$ decline. It is not a directly observed ground-truth metric, but rather a categorized proxy constructed on top of raw Google Search Console impression data.  
- Validation Design & Claim Scope: The observational comparison evaluates $74,187$ rising pages against $45,272$ falling pages. While the sample size is large and establishes a clear statistical correlation, the validation design does not fully carry a causal claim. Because word count and content age are observed concurrently, the data shows that longer, newer pages are currently growing, but it does not prove that adding word length to a declining page will cause its impressions to reverse.

Finding 2: Content Lifecycle Curve (The Content Performance Peak & Decay)

- Paper Finding: Content health peaks between 61–90 days (Health Score $33.1$), enters a maturation plateau, and hits a "decay cliff" at 271–365 days where the score drops to $14$. Performance rebounds after 365+ days ($25.1$), but primarily for older pages that were updated.  
- Label Origin: The outcome label for this finding is a derived proprietary composite index. The FlyRank Health Score (0–100) is constructed by summing weighted points across four sub-metrics: Impressions ($30\text{ pts}$), Position ($30\text{ pts}$), Click-Through Rate ($20\text{ pts}$), and Scroll Depth ($20\text{ pts}$).  
- Validation Design & Claim Scope: The paper uses cross-sectional age buckets across a static cached snapshot of $341,701$ content pieces. While the cross-sectional data clearly demonstrates that 9–12 month old content currently scores lower than 2–3 month old content, a cross-sectional snapshot carries inherent cohort bias. Evaluating individual cohorts longitudinally across time (following the exact same pages over a 12-month lifecycle) would better confirm whether content naturally follows this decay curve versus older cohort pages simply being constructed differently at publication.   

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest validation

In Week 5, the model was evaluated using a regular train/test split.

For this audit, I re-ran the model using a client-grouped split. This prevents rows from the same client from appearing in both training and test data.

The goal is to measure whether the model's ranking performance is also observed on clients that were not present during training.

I compare the earlier measured result with the grouped result. Any difference is treated as an observed validation gap, not as proof that one evaluation is wrong.

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/dee0742/ML-FlyRank-Task/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [22]:
df["is_declining_label"] = (
    df["trend_direction"] == "declining"
).astype(int)

target = "is_declining_label"

excluded = [
    target,
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

feature_cols = [
    c for c in df.columns
    if c not in excluded
]

X = df[feature_cols]
y = df[target]
groups = df["client_id"]

print("Base rate:", y.mean())
print("Features:", len(feature_cols))

Base rate: 0.0
Features: 40


In [23]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=50,
    test_size=0.20,
    random_state=42
)

for train_idx, test_idx in gss.split(X, y, groups):
    if (
        y.iloc[train_idx].nunique() == 2
        and y.iloc[test_idx].nunique() == 2
    ):
        break

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Overlapping clients:", len(train_clients & test_clients))

Train rows: 26796
Test rows: 3204
Overlapping clients: 0


In [26]:
print("Clients with at least one declining row:")
print((client_decline > 0).sum())

print("\nTotal clients:")
print(df["client_id"].nunique())

Clients with at least one declining row:
0

Total clients:
32


In [25]:
print("Trend direction values:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nTarget values:")
print(df["is_declining_label"].value_counts(dropna=False))

print("\nDeclining rows by client:")
client_decline = (
    df.groupby("client_id")["is_declining_label"]
      .sum()
      .sort_values(ascending=False)
)

print(client_decline)

Trend direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Target values:
is_declining_label
0    30000
Name: count, dtype: int64

Declining rows by client:
client_id
client_02d20bbd7e    0
client_0b918943df    0
client_19581e27de    0
client_1a6562590e    0
client_25fc0e7096    0
client_2c624232cd    0
client_349c41201b    0
client_3fdba35f04    0
client_434c9b5ae5    0
client_4e07408562    0
client_4ec9599fc2    0
client_4fc82b26ae    0
client_6208ef0f77    0
client_624b60c58c    0
client_7f2253d7e2    0
client_8527a891e2    0
client_8722616204    0
client_8b940be7fb    0
client_9400f1b21c    0
client_98a3ab7c34    0
client_9f14025af0    0
client_a88a7902cb    0
client_b4944c6ff0    0
client_bbb965ab0c    0
client_bdd2d3af3a    0
client_d029fa3a95    0
client_d4735e3a26    0
client_d59eced1de    0
client_e29c9c180c    0
client_e629fa6598    0
client_f369cb89fc    0
client_f74efabef1    0
Name: is_

In [27]:
print(df["trend_direction"].value_counts(dropna=False))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [28]:
print(df["is_declining_label"].value_counts(dropna=False))

is_declining_label
0    30000
Name: count, dtype: int64


In [29]:
df["trend_direction"] == "declining"

,trend_direction
0,False
1,False
2,False
3,False
4,False
...,...
29995,False
29996,False
29997,False
29998,False


In [30]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [31]:
target = "is_declining_label"

excluded = [
    target,
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

feature_cols = [
    c for c in df.columns
    if c not in excluded
]

X = df[feature_cols]
y = df[target]
groups = df["client_id"]

print("Base rate:", y.mean())
print("Number of features:", len(feature_cols))

Base rate: 0.5420666666666667
Number of features: 40


In [32]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=50,
    test_size=0.20,
    random_state=42
)

for train_idx, test_idx in gss.split(X, y, groups):
    if (
        y.iloc[train_idx].nunique() == 2
        and y.iloc[test_idx].nunique() == 2
    ):
        break

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train classes:", y_train.value_counts().to_dict())
print("Test classes:", y_test.value_counts().to_dict())

Train rows: 23837
Test rows: 6163
Train classes: {1: 13113, 0: 10724}
Test classes: {1: 3149, 0: 3014}


In [33]:
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Overlapping clients:", len(train_clients & test_clients))

assert len(train_clients & test_clients) == 0

Training clients: 25
Test clients: 7
Overlapping clients: 0


In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

test_scores = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")

Model trained successfully.


In [35]:
import numpy as np

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    top_k = order[:k]
    return y_true.iloc[top_k].mean()

y_test_reset = y_test.reset_index(drop=True)
scores_series = pd.Series(test_scores)

honest_p20 = precision_at_k(y_test_reset, scores_series, 20)
honest_p50 = precision_at_k(y_test_reset, scores_series, 50)
honest_p100 = precision_at_k(y_test_reset, scores_series, 100)

honest_base_rate = y_test.mean()

print("Precision@20:", honest_p20)
print("Precision@50:", honest_p50)
print("Precision@100:", honest_p100)
print("Base rate:", honest_base_rate)

Precision@20: 1.0
Precision@50: 1.0
Precision@100: 1.0
Base rate: 0.5109524582184002


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

test_scores = model.predict_proba(X_test)[:, 1]

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I checked the final feature set for three main forms of leakage: label-derived features, future or overlapping information, and decision-derived features.

The target is `is_declining_label`. Since this label is derived from `trend_direction`, and `trend_direction` is derived from `trend_pct`, neither of those columns is included in the feature set.

Client and content IDs are also excluded from the model. `client_id` is used only to create the grouped validation split.

For each remaining feature, the intended question is: could this value have been known at the time the prediction was made?

In [37]:
leakage_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

present_leakage_columns = [
    c for c in leakage_columns
    if c in feature_cols
]

print("Potential leakage columns found in features:")
print(present_leakage_columns)

assert "is_declining_label" not in feature_cols
assert "trend_direction" not in feature_cols
assert "trend_pct" not in feature_cols
assert "content_id" not in feature_cols
assert "client_id" not in feature_cols

print("Leakage-column checks passed.")

Potential leakage columns found in features:
[]
Leakage-column checks passed.


In [38]:
print("Number of features:", len(feature_cols))
print("\nFinal features:")
for col in feature_cols:
    print("-", col)

Number of features: 40

Final features:
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Earlier claim

"Our model accurately predicts which content items are declining."

### Evidence-safe claim

"On the evaluated dataset, the model showed measured ranking performance for identifying items with the observed decline label. Under client-grouped validation, the measured performance was [X], so the result should be treated as directional decision-support rather than proof of performance on all future clients."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- [x] Two research-paper findings were reviewed with constructive methodology questions.
- [x] The Week-5 model was re-run using a client-grouped split.
- [x] Random-split and grouped-split results are compared.
- [x] Base rate is reported.
- [x] Leakage columns were checked.
- [x] `trend_direction` and `trend_pct` are excluded from features.
- [x] `client_id` and `content_id` are excluded from features.
- [x] Real model failures are inspected.
- [x] My strongest claim was rewritten using evidence-safe language.
- [x] The notebook runs from top to bottom without errors.
- [x] No client names, private queries, or private URLs are included.